# Bag of Words Embeddings
### *The Hundred-Page Language Models Book*, Andriy Burkov
### Section 2.1

In [1]:
import re, torch, torch.nn as nn
torch.manual_seed(42)

In [2]:
# define some text and labels

corpus = [
    "movies are fun for everyone", # 0
    "everyone must watch this interesting movie", # 0
        "watching movies is great fun.", # 0
        "Enjoy a great movie today", # 0
        "science research is interesting and important", # 1
        "learning maths is very important for everyone", # 1
        "science discovery is interesting", # 1
        "Maths maths maths science", # 1
        "rock is great to listen to", # 2
        "listen to music for fun", # 2
        "music is fun for everyone", # 2
        "Listen to folk music!" # 2
        ]

#labels=[1,1,1,3,3,3,2,2,2,2,3]
labels=[0,0,0,0,1,1,1,1,2,2,2,2,]
# 0. movies
# 1. science
# 2. music

# check that we have the correct number of labels
print(len(corpus) == len(labels))

num_classes=len(set(labels))
print(num_classes)

True
3


In [3]:
def tokenize(text):
  return re.findall(r"\w+", text.lower())

def get_vocabulary(texts):
  tokens={token for text in texts for token in tokenize(text)}
  return {word: idx for idx, word in enumerate(sorted(tokens))}

vocabulary=get_vocabulary(corpus)
print(type(vocabulary))
print(len(vocabulary))
print(vocabulary)

<class 'dict'>
29
{'a': 0, 'and': 1, 'are': 2, 'discovery': 3, 'enjoy': 4, 'everyone': 5, 'folk': 6, 'for': 7, 'fun': 8, 'great': 9, 'important': 10, 'interesting': 11, 'is': 12, 'learning': 13, 'listen': 14, 'maths': 15, 'movie': 16, 'movies': 17, 'music': 18, 'must': 19, 'research': 20, 'rock': 21, 'science': 22, 'this': 23, 'to': 24, 'today': 25, 'very': 26, 'watch': 27, 'watching': 28}


In [4]:
this_text = "Tokenize this text, please!"
print(this_text)
print(tokenize(this_text))

Tokenize this text, please!
['tokenize', 'this', 'text', 'please']


In [5]:
tokens={token for text in corpus for token in tokenize(text)}
print(type(tokens))
print(len(tokens))
print(tokens)

<class 'set'>
29
{'important', 'music', 'science', 'is', 'for', 'fun', 'learning', 'discovery', 'great', 'this', 'listen', 'are', 'folk', 'and', 'everyone', 'movie', 'research', 'must', 'watching', 'a', 'rock', 'very', 'to', 'movies', 'today', 'watch', 'enjoy', 'maths', 'interesting'}


In [6]:
# convert to Bag of Words
def doc_to_bow(doc, vocabulary):
  """
  inputs:
      doc : str
      vocabulary : dict
  outputs:
      bow : list
  """
  tokens = tokenize(doc) #set(tokenize(doc))
  bow = [0]*len(vocabulary)
  for token in tokens:
    if token in vocabulary:
      bow[vocabulary[token]]+=1
  return bow

In [7]:
# sanity check
v = {'a' : 0, 'b' : 1, 'c' : 2}
doc = "b a a b z"
xx = doc_to_bow(doc,v)
print(xx)
tt = tokenize(doc)
print(type(tt),tt)
ss = set(tt)
print(ss)

[2, 2, 0]
<class 'list'> ['b', 'a', 'a', 'b', 'z']
{'a', 'z', 'b'}


In [8]:
bow_embedding_vectors=torch.tensor(
    [doc_to_bow(doc,vocabulary) for doc in corpus],
    dtype=torch.float32
)

labels=torch.tensor(labels, dtype=torch.long)

In [9]:
print(bow_embedding_vectors.shape)
print(labels.shape)
print(labels)

torch.Size([12, 29])
torch.Size([12])
tensor([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2])


In [10]:
print(torch.sum(bow_embedding_vectors, axis = 0))
print(torch.sum(bow_embedding_vectors, axis = 1))
doc_num = 7
print(corpus[doc_num])
print(bow_embedding_vectors[doc_num,])

tensor([1., 1., 1., 1., 1., 4., 1., 4., 4., 3., 2., 3., 6., 1., 3., 4., 2., 2.,
        3., 1., 1., 1., 3., 1., 4., 1., 1., 1., 1.])
tensor([5., 6., 5., 5., 6., 7., 4., 4., 6., 5., 5., 4.])
Maths maths maths science
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 3., 0., 0.,
        0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.])


In [11]:
print(torch.max(bow_embedding_vectors))

tensor(3.)


In [12]:
input_dim=len(vocabulary)
hidden_dim=50
output_dim=num_classes
print("num_classes = ", num_classes)

class SimpleClassifier(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super().__init__() # runs the constructor of the parent class, nn.Module
    self.fc1=nn.Linear(input_dim, hidden_dim)
    self.relu=nn.ReLU()
    self.fc2=nn.Linear(hidden_dim, output_dim)
    # No final softmax layer : Pytorch's CrossEntropyLoss combines softmax and cross entropy loss internally. 

  def forward(self, x):
    x=self.fc1(x)
    x=self.relu(x)
    x=self.fc2(x)
    return x

model=SimpleClassifier(input_dim,hidden_dim,output_dim)

num_classes =  3


In [13]:
print(model)

SimpleClassifier(
  (fc1): Linear(in_features=29, out_features=50, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=50, out_features=3, bias=True)
)


In [14]:
print("W1.shape = ", model.fc1.weight.shape)   # shape (50, 26)
print("b1.shape = ", model.fc1.bias.shape)     # shape (50,)
print("W2.shape = ", model.fc2.weight.shape)   # shape (11, 50)
print("b2.shape = ", model.fc2.bias.shape)     # shape (11,)
print("b2 = ", model.fc2.bias)

W1.shape =  torch.Size([50, 29])
b1.shape =  torch.Size([50])
W2.shape =  torch.Size([3, 50])
b2.shape =  torch.Size([3])
b2 =  Parameter containing:
tensor([-0.0048, -0.0628,  0.0465], requires_grad=True)


In [15]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(model.parameters(), lr=0.001)

num_steps = 3001

for step in range(num_steps):
  optimizer.zero_grad()
  loss=criterion(model(bow_embedding_vectors), labels)
  loss.backward()
  optimizer.step()
  if step % 100 == 0:
      print(step)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000


In [16]:
print(type(model.parameters()))
print(model.parameters())

<class 'generator'>
<generator object Module.parameters at 0x000001CF0F3811C0>


In [17]:
class_names=["movies","science","music"]

# evaluate on the training data:
with torch.no_grad():
  outputs=model(bow_embedding_vectors)
  print(type(outputs), outputs.shape)
  predicted_ids=torch.argmax(outputs,dim=1)
  print(predicted_ids)

for i, doc in enumerate(corpus):
  print(f'{doc}:{class_names[predicted_ids[i].item()]}')


<class 'torch.Tensor'> torch.Size([12, 3])
tensor([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2])
movies are fun for everyone:movies
everyone must watch this interesting movie:movies
watching movies is great fun.:movies
Enjoy a great movie today:movies
science research is interesting and important:science
learning maths is very important for everyone:science
science discovery is interesting:science
Maths maths maths science:science
rock is great to listen to:music
listen to music for fun:music
music is fun for everyone:music
Listen to folk music!:music


In [18]:
new_corpus=["listening to rock music is fun", 
            "I love science and maths", 
            "cinema is the best", 
            "movies", 
            "cinema", 
            "cinematography", 
            "science music movies", 
            "science movies",
            "cinema", 
            "music", 
            "science", 
            "maths", 
            "math", 
            "mathematics", 
            "algebra"]

# # 0. movies
# 1. science
# 2. music

In [19]:
new_bow_embedding_vectors=torch.tensor([doc_to_bow(new_doc, vocabulary) for new_doc in new_corpus], dtype=torch.float32)

In [20]:
print(new_bow_embedding_vectors.shape)

torch.Size([15, 29])


In [21]:
with torch.no_grad():
  outputs=model(new_bow_embedding_vectors)
  print(type(outputs), outputs.shape)
  predicted_ids=torch.argmax(outputs,dim=1)
  print(predicted_ids)

for i, new_doc in enumerate(new_corpus):
  print(f'{new_doc}:{class_names[predicted_ids[i].item()]}')

<class 'torch.Tensor'> torch.Size([15, 3])
tensor([2, 1, 1, 0, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1, 1])
listening to rock music is fun:music
I love science and maths:science
cinema is the best:science
movies:movies
cinema:science
cinematography:science
science music movies:music
science movies:science
cinema:science
music:music
science:science
maths:science
math:science
mathematics:science
algebra:science


<b>Exercise:</b> Why does the model mis-predict "cinema" ? How could we address this?

In [22]:
new_doc = "algebra"
new_doc_vector=torch.tensor([doc_to_bow(new_doc, vocabulary)], dtype=torch.float32)
print(doc_to_bow(new_doc, vocabulary))
print(new_doc_vector)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0.]])
